# ⚽ Getting Started with Soccer Analytics Visualization

This notebook demonstrates how to use the soccer analytics visualization toolkit to create Instagram-ready content using StatsBomb data.

## 1. Setup and Imports

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

# Core imports
from src.data.loader import StatsBombLoader, COMPETITIONS
from src.visualizations.pitch import SoccerViz
from src.animations.match_animation import MatchAnimator, GoalSequenceAnimator
from src.export.instagram import InstagramExporter

# Additional imports
import pandas as pd
import matplotlib.pyplot as plt

# For inline display
%matplotlib inline

print("✅ Imports successful!")

## 2. Load Data

In [ ]:
# Initialize the data loader
loader = StatsBombLoader()

# View available competitions
competitions = loader.get_competitions()
print(f"Found {len(competitions)} competitions")
competitions.head(10)

In [ ]:
# Load World Cup 2022 matches
# Competition ID 43 = FIFA World Cup, Season ID 106 = 2022
matches = loader.get_matches(competition_id=43, season_id=106)
print(f"Found {len(matches)} matches")
matches[['match_id', 'home_team', 'away_team', 'home_score', 'away_score']].head(10)

In [ ]:
# Pick a specific match - let's find Argentina matches
argentina_matches = matches[
    (matches['home_team'] == 'Argentina') | 
    (matches['away_team'] == 'Argentina')
]
argentina_matches[['match_id', 'home_team', 'away_team', 'home_score', 'away_score']]

In [ ]:
# Load events for a specific match
# Replace with an actual match_id from above
match_id = argentina_matches.iloc[0]['match_id']
events = loader.get_match_events(match_id, split=True)

print(f"Loaded {len(events)} events")
print(f"\nEvent types: {events['type'].unique()}")

## 3. Create Static Visualizations

In [ ]:
# Initialize the visualizer
viz = SoccerViz()

# Get team names from the match
teams = events['team'].unique()
print(f"Teams: {teams}")

### 3.1 Shot Map

In [ ]:
# Create shot map for first team
team_name = teams[0]
fig, ax = viz.plot_shot_map(events, team_name=team_name)
plt.show()

### 3.2 Pass Map

In [ ]:
# Create pass map for a team
fig, ax = viz.plot_pass_map(events, team_name=team_name)
plt.show()

### 3.3 Heatmap

In [ ]:
# Create heatmap showing team activity
fig, ax = viz.plot_heatmap(events, team_name=team_name)
plt.show()

## 4. Create Animations

In [ ]:
# Create animator with events
animator = MatchAnimator(events, fps=5)

# Create pass flow animation for first 15 minutes
anim = animator.create_pass_flow_animation(
    team_name=team_name,
    time_range=(0, 15)
)

# Display in notebook (may require additional setup)
from IPython.display import HTML
HTML(anim.to_jshtml())

In [ ]:
# Save animation as GIF
output_path = animator.save_gif('pass_flow.gif', output_dir='../output/gifs')
print(f"Saved to: {output_path}")

## 5. Export for Instagram

In [ ]:
# Initialize exporter
exporter = InstagramExporter(output_dir='../output')

# Create a visualization and export as Instagram post
fig, ax = viz.plot_shot_map(events, team_name=team_name)

output_path = exporter.export_post(
    fig,
    'shot_map_instagram.png',
    format='square',
    add_watermark=True,
    watermark_text='@your_soccer_analytics'
)

print(f"Exported to: {output_path}")

## 6. Goal Sequence Animation

In [ ]:
# Find goals in the match
goal_animator = GoalSequenceAnimator(events)
goals = goal_animator.get_goals()

if len(goals) > 0:
    print(f"Found {len(goals)} goal(s)")
    print(goals[['minute', 'player', 'team', 'shot_statsbomb_xg']])
else:
    print("No goals found in this match")

In [ ]:
# Animate the first goal with buildup
if len(goals) > 0:
    goal_idx = goals.index[0]
    anim = goal_animator.animate_goal(
        goal_index=goal_idx,
        num_buildup_events=8,
        fps=3
    )
    HTML(anim.to_jshtml())

## 🎉 Next Steps

- Explore different matches and competitions
- Customize colors and styling in `config/settings.ini`
- Try player-specific visualizations
- Experiment with different animation styles
- Set up Instagram automation (coming soon!)